In [ ]:
///- --package . --timeout 1200000
:< sm'_operators
:< json_schema

:| pint_id =
    | PintTerm : i32 * option i32
    | PintGroup : i32

:== pint_row =
    {
        id : option pint_id
        name : string
        slug : string
        own : canon::cardinality
        semantic : option canon::cardinality
        effective : canon::cardinality
        path : string
    }

:= after (marker : string) (text : string) : string =
    := at = sm'::index_of marker text
    := cut = at + sm'::length marker
    ? at < 0i32 ?> "" |: sm'::substring cut (sm'::length text - cut) text

:= before (marker : string) (text : string) : string =
    := at = sm'::index_of marker text
    ? at < 0i32 ?> text |: sm'::substring 0i32 at text

:= words (text : string) : string =
    := spaced = sm'::replace "\t" " " (sm'::replace "\n" " " (sm'::replace "\r" " " text))
    sm'::concat_list " " (listm'::filter (\ (w : string) => w <> "") (sm'::split_list " " spaced))

:= entities_decoded (text : string) : string = sm'::replace "&#34;" "\"" (sm'::replace "&#39;" "'" text)

:= tag_tail (part : string) : string = ? sm'::contains ">" part ?> after ">" part |: part

:= untagged (html : string) : string =
    words (entities_decoded (sm'::concat_list "" (listm::map tag_tail (sm'::split_list "<" html))))

:= cell_text (piece : string) : string = untagged (before "</td>" (after ">" piece))

:= row_cells (row : string) : list string =
    ?= sm'::split_list "<td" row ?|
    | _ :: cells => listm::map cell_text cells
    | [] => []

:= sub_term (parts : list string) : option i32 =
    ?= parts ?|
    | _ :: sub :: [] => sm'::decimal_i32 sub
    | _ => None

:= term_of (rest : string) : option pint_id =
    := parts = sm'::split_list "-" rest
    ?= sm'::decimal_i32 (before "-" rest) ?|
    | Some n ?& listm::length parts <= 2i32 => Some (PintTerm (n, sub_term parts))
    | _ => None

:= group_of (rest : string) : option pint_id =
    ?= sm'::decimal_i32 rest ?|
    | Some n => Some (PintGroup n)
    | None => None

:= pint_id_of (text : string) : option pint_id =
    ? sm'::starts_with "IBT-" text
    ?> term_of (after "IBT-" text)
    |? sm'::starts_with "IBG-" text
    ?> group_of (after "IBG-" text)
    |: None

:= pint_id_text (id : pint_id) : string =
    ?= id ?|
    | PintTerm (n, None) => "IBT-" ++# sm'::pad_left 3 '0' (sm'::obj_to_string n)
    | PintTerm (n, Some s) => "IBT-" ++# sm'::pad_left 3 '0' (sm'::obj_to_string n) ++# "-" ++# sm'::obj_to_string s
    | PintGroup n => "IBG-" ++# sm'::pad_left 2 '0' (sm'::obj_to_string n)

:= semantic_of (text : string) : option canon::cardinality = ? text = "" ?> None |: Some (canon::cardinality_of text)

:= parent_slug (slug : string) : string =
    := at = sm'::last_index_of "/" slug
    ? at < 0i32 ?> "" |: sm'::substring 0i32 at slug

:= is_prefixed (step : string) : bool = sm'::length step > 4i32 && sm'::substring 3i32 1i32 step = "-"

:= variant_suffix (variant : string) : string = ? variant = "" ?> "" |: "[" ++# variant ++# "]"

:= base_step (step : string) : string = sm'::substring 0i32 3i32 step ++# ":" ++# before "-" (after "-" step)

:= prefixed_step (step : string) : string = base_step step ++# variant_suffix (after "-" (after "-" step))

:= step_text (step : string) : string = ? is_prefixed step ?> prefixed_step step |: "@" ++# step

:= qualifier_of (element : string) : string = ? sm'::contains "[" element ?> "[" ++# after "[" element |: ""

:= own_step_text (step : string) (qualifier : string) : string =
    ? qualifier <> "" && is_prefixed step ?> base_step step ++# qualifier |: step_text step

:= last_step (slug : string) : string =
    := at = sm'::last_index_of "/" slug
    ? at < 0i32 ?> slug |: sm'::substring (at + 1i32) (sm'::length slug - at - 1i32) slug

:= parent_of (slug : string) (seen : list pint_row) : option pint_row =
    listm'::try_find_ (\ (r : pint_row) => r::slug = parent_slug slug) seen

:= parent_effective (parent : option pint_row) : canon::cardinality =
    ?= parent ?|
    | Some p => p::effective
    | None => canon::exactly_once ()

:= parent_path (parent : option pint_row) : string =
    ?= parent ?|
    | Some p => p::path ++# "/"
    | None => ""

:== row_cells_of = { id : string; semantic : string; name : string; own : string; element : string }

:= row_path (slug : string) (parent : option pint_row) (c : row_cells_of) : string =
    parent_path parent ++# own_step_text (last_step slug) (qualifier_of c::element)

:= row_effective (parent : option pint_row) (c : row_cells_of) : canon::cardinality =
    canon::within (parent_effective parent) (canon::cardinality_of c::own)

:= row_record (slug : string) (parent : option pint_row) (c : row_cells_of) : pint_row =
    {
        id = pint_id_of c::id
        name = c::name
        slug = slug
        own = canon::cardinality_of c::own
        semantic = semantic_of c::semantic
        effective = row_effective parent c
        path = row_path slug parent c
    }

:= cells_of (cells : list string) : option row_cells_of =
    ?= cells ?|
    | _ :: id :: semantic :: name :: own :: element :: _ =>
        Some { id = id; semantic = semantic; name = name; own = own; element = element }
    | _ => None

:= row_of (chunk : string) (seen : list pint_row) : option pint_row =
    := row = before "</tr>" chunk
    := slug = before "/\"" (after "/syntax/" row)
    ?= cells_of (row_cells row) ?|
    | Some c => Some (row_record slug (parent_of slug seen) c)
    | None => None

:~ ^ rows_from (chunks : list string) (acc : list pint_row) : list pint_row =
    ?= chunks ?|
    | chunk :: rest =>
        ?= row_of chunk acc ?|
        | Some r => rows_from rest (r :: acc)
        | None => rows_from rest acc
    | [] => listm::rev acc

:= syntax_rows (html : string) : list pint_row =
    ?= sm'::split_list "<tr id='SB-" html ?|
    | _ :: chunks => rows_from chunks []
    | [] => []

:= terms_of (rows : list pint_row) : list (pint_id * pint_row) =
    listm::concat (listm::map (\ (r : pint_row) => ?= r::id ?| Some id => [ (id, r) ] | None => []) rows)

:= ubl_path_of (root : string) (r : pint_row) : string = root ++# "/" ++# r::path

:= is_base_term (id : pint_id) : bool =
    ?= id ?|
    | PintTerm (_, None) => !+
    | _ => !-

:= base_number (id : pint_id) : option i32 =
    ?= id ?|
    | PintTerm (n, None) => Some n
    | _ => None

:= is_addition (bound : list canon::term_id) (id : pint_id) : bool =
    ?= base_number id ?|
    | Some n => listm'::contains (canon::BusinessTerm n) bound = !-
    | None => !-

:= transaction_of (document : canon::ubl_document) : string = "trn-" ++# case_labels::case_label "Ubl" "" document

:= root_of (document : canon::ubl_document) : string =
    "ubl:" ++# case_labels::unprefixed "Ubl" (reflection::case_name document)

:= pint_page (document : canon::ubl_document) : string =
    := file = "deps/corpus/eu/peppol-pint/syntax/" ++# transaction_of document ++# ".html"
    file_system::read_all_text (fiscal_paths::under file)

:= document_terms (document : canon::ubl_document) : list (pint_id * pint_row) =
    terms_of (syntax_rows (pint_page document))

:= has_id (id : pint_id) (terms : list (pint_id * pint_row)) : bool =
    listm::length (listm'::filter (\ (i, _) => i = id) terms) > 0i32

:~ ^ merged_terms (terms : list (pint_id * pint_row)) (acc : list (pint_id * pint_row)) : list (pint_id * pint_row) =
    ?= terms ?|
    | (id, r) :: rest => merged_terms rest (? has_id id acc ?> acc |: listm::append acc [ (id, r) ])
    | [] => acc

:= binding_object (root : string) (r : pint_row) : json_value =
    := own = canon::cardinality_text r::own
    := effective = canon::cardinality_text r::effective
    JsonObject [
        ("path", JsonText (ubl_path_of root r))
        ("cardinality", JsonText own)
        ("effective", JsonText effective)
    ]

:= binding_json (document : canon::ubl_document) (r : pint_row) : string * json_value =
    canon::ubl_document_name document, binding_object (root_of document) r

:= bindings
    (id : pint_id) (documents : list (canon::ubl_document * list (pint_id * pint_row)))
    : list (string * json_value)
    =
    := bound ((d, terms) : canon::ubl_document * list (pint_id * pint_row)) : list (string * json_value) =
        listm::map (\ (_, r) => binding_json d r) (listm'::filter (\ (i, _) => i = id) terms)
    listm::concat (listm::map bound documents)

:= semantic_text (r : pint_row) : string =
    ?= r::semantic ?|
    | Some c => canon::cardinality_text c
    | None => ""

:= pint_term_json
    (documents : list (canon::ubl_document * list (pint_id * pint_row))) ((id, r) : pint_id * pint_row)
    : json_value
    =
    := head =
        [ ("id", JsonText (pint_id_text id)); ("name", JsonText r::name); ("semantic", JsonText (semantic_text r)) ]
    JsonObject (listm::append head [ ("ubl", JsonObject (bindings id documents)) ])

:= additions_of (terms : list (pint_id * pint_row)) : list string =
    := bound = listm::map (\ (r : canon::term_row) => r::id) (canon::en16931_terms ())
    listm::map (\ (id, _) => pint_id_text id) (listm'::filter (\ (id, _) => is_addition bound id) terms)

#### a_slug_step_becomes_its_ubl_name (test)

In [1]:
///- --test
///> rust

:= steps = [ "cac-Party"; "cac-TaxTotal-1"; "unitCode"; "cbc-ID" ]
? listm::map step_text steps <> [ "cac:Party"; "cac:TaxTotal[1]"; "@unitCode"; "cbc:ID" ]
?> failwith "a prefixed step keeps its prefix, a variant becomes [n], a bare step is an attribute"
?
    listm::map pint_id_of [ "IBT-034-1"; "IBG-05"; "AUTO-14"; "IBT-034-1-2" ]
    <> [ Some (PintTerm (34i32, Some 1i32)); Some (PintGroup 5i32); None; None ]
?> failwith "IBT ids carry at most one sub-term, IBG ids a group number, AUTO rows none"
? untagged "<div> Seller&#39;s <a href=\"x\">name</a>\n </div>" <> "Seller's name" ?> failwith "cells lose tags"
? (transaction_of canon::UblCreditNote, root_of canon::UblCreditNote) <> ("trn-creditnote", "ubl:CreditNote")
?> failwith "each UBL document names its PINT page and root"

0

#### pint_invoice_terms_are_read_from_the_syntax_page (test)

In [2]:
///- --test
///> rust

:= rows = syntax_rows (pint_page canon::UblInvoice)
:= terms = terms_of rows
:= base = listm'::filter (\ (id, _) => is_base_term id) terms
:= groups = listm'::filter (\ (id, _) => ?= id ?| PintGroup _ => !+ | _ => !-) terms
:= counts : list i32 =
    [
        listm::length rows
        listm::length base
        listm::length terms - listm::length base - listm::length groups
        listm::length groups
    ]
console::write_line (
    "pint invoice rows, terms, sub-terms, groups: " ++# sm'::concat_list ", " (listm::map sm'::obj_to_string counts)
)
? counts <> [ 362i32; 195i32; 24i32; 36i32 ]
?> failwith "362 syntax rows: 195 IBT terms, 24 sub-terms, 36 IBG groups"
:= texts = additions_of terms
? listm::length texts <> 33i32 || listm'::contains "IBT-168" texts = !- || listm'::contains "IBT-130" texts
?> failwith "33 PINT terms have no EN 16931 row in the canon (issue time IBT-168 among them; IBT-130 is BT-130)"

pint invoice rows, terms, sub-terms, groups: 362, 195, 24, 36

0

#### pint_terms_are_written_to_the_canon (test)

In [3]:
///- --test
///> rust

:= documents =
    listm::map (\ (d : canon::ubl_document) => d, document_terms d) [ canon::UblInvoice; canon::UblCreditNote ]
:= merged = merged_terms (listm::concat (listm::map (\ (_, t) => t) documents)) []
:= credit = document_terms canon::UblCreditNote
? (listm::length credit, listm::length merged) <> (250i32, 255i32)
?> failwith "250 credit-note terms, every one also an invoice term, 255 in all"
:= due = listm'::filter (\ (id, _) => id = PintTerm (9i32, None)) credit
:= path_of (r : pint_row) : string = ubl_path_of (root_of canon::UblCreditNote) r
:= binding_of ((_, r) : pint_id * pint_row) : string * string = (path_of r, canon::cardinality_text r::effective)
? listm::map binding_of due <> [ ("ubl:CreditNote/cac:PaymentMeans/cbc:PaymentDueDate", "0..n") ]
?> failwith "a credit note binds the payment due date per payment means, an invoice once at cbc:DueDate"
:= creditor = listm'::filter (\ (id, _) => id = PintTerm (90i32, None)) (document_terms canon::UblInvoice)
:= invoice_path ((_, r) : pint_id * pint_row) : string = ubl_path_of "ubl:Invoice" r
:= sepa (path : string) : bool =
    sm'::contains "cac:Party/cac:PartyIdentification[cac:ID/@schemeID = " path && sm'::contains "SEPA" path
? listm::map (\ (t : pint_id * pint_row) => sepa (invoice_path t)) creditor <> [ !+ ] ?>
    failwith (
        "a variant step carries its printed qualifier, so IBT-090 is the SEPA PartyIdentification, not the "
        ^^ "first one"
    )
:= source = "OpenPeppol PINT Billing 1.1.3 syntax trees, trn-invoice and trn-creditnote"
:= terms = JsonArray (listm::map (pint_term_json documents) merged)
:= additions = JsonArray (listm::map JsonText (additions_of merged))
file_system::directory_create_all (fiscal_paths::under "canon") |> ignore
file_system::write_all_text
    (fiscal_paths::under "canon/pint-terms.json")
    (json_text (JsonObject [ ("source", JsonText source); ("terms", terms); ("additions", additions) ]) ++# "\n")

0